# The `Send` API [Step 07.04 - Dynamic fan-out and map-reduce]

> **MLCourse - Agentic AI - LangGraph**

Everything so far has had a **fixed** shape: you decide at build time how many
branches exist. Conditional edges pick *which* of a known set of nodes runs next.
They cannot create branches that did not exist when you called `compile()`.

`Send` can. It is LangGraph's answer to:

> *"An LLM just told me there are 7 subtopics. Run the research node 7 times, in
> parallel, and collect the results."*

You saw `Send` mentioned in passing in `06_multi_agent_systems/04_parallel_agents`.
This notebook is the real treatment.

### What you'll learn

- Why conditional edges cannot express runtime fan-out.
- `Send(node_name, payload)` - what the payload actually is.
- Giving the worker node its **own input schema** (this is a mini isolated subgraph).
- The **reducer requirement** for the reduce step (`operator.add`).
- Fan-out to *different* nodes, conditional fan-out, and empty fan-out.
- A complete map-reduce with a real LLM deciding the branch count.

### Key takeaways

- `Send` is returned from a **conditional edge function**, as a list.
- Each `Send` carries its own private state dict - it is **not** the parent state.
- The collector node needs a reducer or the parallel writes will conflict.

### Setup: environment, model factory, rate-limit-aware call helper


In [ ]:
import os                                  # environment variable access
import time                                # timing + backoff sleeps
from pathlib import Path                   # locating the track root
from dotenv import load_dotenv             # reads KEY=value pairs from .env

# Walk UP from the notebook folder until we find the track root `03_agentic_ai`,
# then load the (gitignored) .env that lives there. Every provider-touching
# notebook in this track uses exactly this block.
TRACK = Path.cwd()
while TRACK.name != "03_agentic_ai" and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / ".env")

GROQ_KEY = os.getenv("GROQ_API_KEY")       # never print this value
GROQ_MODEL = "qwen/qwen3.8-27b"            # fast hosted model, generous free tier
OLLAMA_MODEL = "llama3.1:8b"               # local fallback if Groq is unavailable


def make_llm(temperature: float = 0.0, max_tokens: int = 512):
    """Return a chat model. Groq first (fast, hosted); local Ollama as fallback.

    OpenAI is never used anywhere in this course.
    """
    if GROQ_KEY:
        from langchain_groq import ChatGroq
        return ChatGroq(model=GROQ_MODEL, api_key=GROQ_KEY,
                        temperature=temperature, max_tokens=max_tokens)
    from langchain_ollama import ChatOllama
    return ChatOllama(model=OLLAMA_MODEL, temperature=temperature)


def safe_invoke(model, messages, retries: int = 4, pause: float = 1.5):
    """Invoke a chat model with exponential backoff on rate limits (HTTP 429).

    Groq's free tier allows roughly 8000 tokens per minute. Teaching notebooks
    fire many small calls in a row, so a retry loop is not optional here.
    """
    delay = pause
    for attempt in range(retries):
        try:
            out = model.invoke(messages)
            time.sleep(pause)              # pace the next call politely
            return out
        except Exception as exc:
            if attempt == retries - 1:
                raise
            print("  [backoff] %s -- retrying in %.1fs" % (type(exc).__name__, delay))
            time.sleep(delay)
            delay *= 2                     # exponential backoff
    raise RuntimeError("unreachable")


print("Track root :", TRACK.name)
print("Provider   :", "Groq / " + GROQ_MODEL if GROQ_KEY else "Ollama / " + OLLAMA_MODEL)


### 1. Why conditional edges are not enough

A conditional edge function returns node *names*. The names must already exist in
the graph. So the maximum number of parallel branches is fixed when you build:

In [2]:
from typing import Annotated, TypedDict
import operator, time
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send


class FixedState(TypedDict):
    n: int
    log: Annotated[list, operator.add]


def worker_a(s): return {"log": ["a ran"]}
def worker_b(s): return {"log": ["b ran"]}


def route(state: FixedState):
    """Returning a LIST of names fans out - but only to nodes that already exist."""
    return ["worker_a", "worker_b"][: state["n"]]


fg = StateGraph(FixedState)
fg.add_node("worker_a", worker_a)
fg.add_node("worker_b", worker_b)
fg.add_conditional_edges(START, route, ["worker_a", "worker_b"])
fg.add_edge("worker_a", END)
fg.add_edge("worker_b", END)
fixed = fg.compile()

print("n=1 ->", fixed.invoke({"n": 1, "log": []})["log"])
print("n=2 ->", fixed.invoke({"n": 2, "log": []})["log"])
print("n=7 -> impossible: there is no worker_c..worker_g to name.")

n=1 -> ['a ran']
n=2 -> ['a ran', 'b ran']
n=7 -> impossible: there is no worker_c..worker_g to name.


### 2. `Send` in one cell

`Send("node_name", {...})` says: *run `node_name` with **this** dict as its state.*

Return a list of them from a conditional edge and you get one parallel branch per
`Send`. The list length is computed at runtime, so it can be anything.

In [3]:
class MapState(TypedDict):
    items: list                                   # the things to map over
    results: Annotated[list, operator.add]        # REDUCER IS MANDATORY here


class ItemState(TypedDict):
    """The worker's OWN schema. A Send payload must satisfy THIS, not MapState."""
    item: str


def process_one(state: ItemState) -> dict:
    """Runs once per Send. It only ever sees {'item': ...}."""
    return {"results": [state["item"].upper()]}


def fan_out(state: MapState):
    """Conditional edge that builds N branches at runtime."""
    return [Send("process_one", {"item": it}) for it in state["items"]]


mg = StateGraph(MapState)
mg.add_node("process_one", process_one)
mg.add_conditional_edges(START, fan_out, ["process_one"])
mg.add_edge("process_one", END)
map_app = mg.compile()

for items in (["a"], ["a", "b", "c"], list("abcdefg")):
    out = map_app.invoke({"items": items, "results": []})
    print("%d items -> %s" % (len(items), out["results"]))

1 items -> ['A']
3 items -> ['A', 'B', 'C']
7 items -> ['A', 'B', 'C', 'D', 'E', 'F', 'G']


Three things to notice:

1. **The worker has its own schema (`ItemState`).** The `Send` payload becomes the
   worker's entire input state. This is a miniature version of the isolated-state
   pattern from notebook 03 - and it is why `process_one` cannot read `items`.
2. **`results` needs `operator.add`.** Without a reducer, several branches writing
   the same key at the same super-step is an error, not a merge (demo below).
3. **The branch count changed per invocation** with no graph changes.

### 3. What happens without a reducer


In [4]:
class NoReducerState(TypedDict):
    items: list
    results: list            # <-- NO reducer. Parallel writes will conflict.


def process_bad(state: ItemState) -> dict:
    return {"results": [state["item"]]}


nr = StateGraph(NoReducerState)
nr.add_node("process_bad", process_bad)
nr.add_conditional_edges(START, lambda s: [Send("process_bad", {"item": i}) for i in s["items"]],
                         ["process_bad"])
nr.add_edge("process_bad", END)

try:
    print(nr.compile().invoke({"items": ["a", "b", "c"], "results": []}))
except Exception as exc:
    print("ERROR ->", type(exc).__name__)
    print(str(exc)[:300])
    print()
    print("Fix: annotate the collector key, e.g.")
    print("     results: Annotated[list, operator.add]")

ERROR -> InvalidUpdateError
At key 'results': Can receive only one value per step. Use an Annotated key to handle multiple values.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/INVALID_CONCURRENT_GRAPH_UPDATE

Fix: annotate the collector key, e.g.
     results: Annotated[list, operator.add]


> **Pitfall:** with a single branch this bug hides - one writer never conflicts.
> It appears the first time a user's request produces two subtopics. Always add
> the reducer, even while testing with one item.

### 4. Parallelism is real

The branches run concurrently within one super-step. Here is a timing proof with
a deliberately slow worker.

In [5]:
class TimedState(TypedDict):
    items: list
    results: Annotated[list, operator.add]


def slow_worker(state: ItemState) -> dict:
    time.sleep(0.4)                     # simulate an API call
    return {"results": [state["item"]]}


tg = StateGraph(TimedState)
tg.add_node("slow_worker", slow_worker)
tg.add_conditional_edges(START, lambda s: [Send("slow_worker", {"item": i}) for i in s["items"]],
                         ["slow_worker"])
tg.add_edge("slow_worker", END)
timed = tg.compile()

n = 5
t0 = time.time()
timed.invoke({"items": list("abcde"), "results": []})
elapsed = time.time() - t0
print("%d workers x 0.4s each" % n)
print("sequential would take : %.1fs" % (n * 0.4))
print("actual wall clock     : %.2fs" % elapsed)

5 workers x 0.4s each
sequential would take : 2.0s
actual wall clock     : 0.40s


### 5. Fan-out to *different* nodes

The list can mix node names. This is how you dispatch heterogeneous work -
route each item to the specialist that fits it.

In [6]:
class RouteState(TypedDict):
    tasks: list
    done: Annotated[list, operator.add]


class TaskState(TypedDict):
    text: str


def handle_number(s: TaskState): return {"done": ["number handler: " + s["text"]]}
def handle_word(s: TaskState):   return {"done": ["word handler  : " + s["text"]]}


def dispatch(state: RouteState):
    sends = []
    for t in state["tasks"]:
        target = "handle_number" if t.isdigit() else "handle_word"
        sends.append(Send(target, {"text": t}))
    return sends


rg = StateGraph(RouteState)
rg.add_node("handle_number", handle_number)
rg.add_node("handle_word", handle_word)
rg.add_conditional_edges(START, dispatch, ["handle_number", "handle_word"])
rg.add_edge("handle_number", END)
rg.add_edge("handle_word", END)

for line in rg.compile().invoke({"tasks": ["42", "banana", "7", "kiwi"], "done": []})["done"]:
    print(" ", line)

  number handler: 42
  word handler  : banana
  number handler: 7
  word handler  : kiwi


### 6. The empty fan-out

If the list is empty, **no branch runs**. If every path out of `START` is an empty
`Send` list, the graph simply ends. Guard for this - an empty plan usually means
your planner failed, and silently producing an empty result is worse than raising.

In [7]:
print("empty items ->", map_app.invoke({"items": [], "results": []}))
print()
print("No error, no workers, empty results. In production, validate the plan")
print("BEFORE fanning out and raise (or route to a recovery node) if it is empty.")

empty items -> {'items': [], 'results': []}

No error, no workers, empty results. In production, validate the plan
BEFORE fanning out and raise (or route to a recovery node) if it is empty.


### 7. The real thing: LLM-planned map-reduce

Now the pattern that actually earns its keep.

- **Plan** - the model reads a topic and decides how many subtopics there are.
  We do not know the count in advance.
- **Map** - one `Send` per subtopic; each branch makes its own LLM call in parallel.
- **Reduce** - a final node synthesises all the branch results.

This is the classic map-reduce summarisation shape, and it is the single most
common real use of `Send`.

In [8]:
import re
from langgraph.graph import StateGraph, START, END


class ResearchState(TypedDict):
    topic: str
    subtopics: list
    findings: Annotated[list, operator.add]     # reducer collects parallel writes
    report: str


class SubtopicState(TypedDict):
    """Worker schema - exactly what each Send payload carries."""
    topic: str
    subtopic: str


planner_llm = make_llm(max_tokens=120)
worker_llm = make_llm(max_tokens=110)
writer_llm = make_llm(max_tokens=260)


def plan(state: ResearchState) -> dict:
    """MAP-PLAN: the model decides the branch count."""
    prompt = (
        "Break the topic below into 3 to 5 distinct subtopics.\n"
        "Output ONLY the subtopics, one per line, no numbering, no extra text.\n\n"
        "Topic: " + state["topic"]
    )
    raw = safe_invoke(planner_llm, prompt).content
    subs = [re.sub(r"^[\s\-\*\d\.\)]+", "", ln).strip() for ln in raw.splitlines() if ln.strip()]
    subs = [s for s in subs if 3 < len(s) < 200][:5]
    print("planner produced %d subtopics:" % len(subs))
    for s in subs:
        print("   -", s)
    return {"subtopics": subs}


def fan_out_research(state: ResearchState):
    """MAP: one branch per subtopic. Count known only now, at runtime."""
    if not state["subtopics"]:
        raise ValueError("planner returned no subtopics - refusing to fan out")
    return [Send("research_one", {"topic": state["topic"], "subtopic": s})
            for s in state["subtopics"]]


def research_one(state: SubtopicState) -> dict:
    """MAP-WORKER: runs once per subtopic, in parallel with its siblings."""
    prompt = ("In at most 2 sentences, explain '%s' in the context of '%s'."
              % (state["subtopic"], state["topic"]))
    text = safe_invoke(worker_llm, prompt).content.strip()
    return {"findings": ["## %s\n%s" % (state["subtopic"], text)]}


def reduce_report(state: ResearchState) -> dict:
    """REDUCE: one pass over everything the branches produced."""
    joined = "\n\n".join(state["findings"])
    prompt = ("Write a 3-sentence executive summary of these research notes.\n\n"
              + joined)
    return {"report": safe_invoke(writer_llm, prompt).content.strip()}


rg = StateGraph(ResearchState)
rg.add_node("plan", plan)
rg.add_node("research_one", research_one)
rg.add_node("reduce_report", reduce_report)
rg.add_edge(START, "plan")
rg.add_conditional_edges("plan", fan_out_research, ["research_one"])
rg.add_edge("research_one", "reduce_report")
rg.add_edge("reduce_report", END)
research_app = rg.compile()

print(research_app.get_graph().draw_ascii())

  +-----------+    
  | __start__ |    
  +-----------+    
        *          
        *          
        *          
    +------+       
    | plan |       
    +------+       
        .          
        .          
        .          
+--------------+   
| research_one |   
+--------------+   
        *          
        *          
        *          
+---------------+  
| reduce_report |  
+---------------+  
        *          
        *          
        *          
   +---------+     
   | __end__ |     
   +---------+     


In [9]:
t0 = time.time()
out = research_app.invoke({
    "topic": "how retrieval-augmented generation works",
    "subtopics": [], "findings": [], "report": "",
})
elapsed = time.time() - t0

print()
print("branches run :", len(out["findings"]))
print("wall clock   : %.1fs" % elapsed)
print()
print("=== FINDINGS ===")
for f in out["findings"]:
    print(f)
    print()
print("=== REPORT ===")
print(out["report"])

planner produced 5 subtopics:
   - Retrieval mechanisms and vector database indexing
   - Query transformation and semantic search strategies
   - Context assembly and prompt construction
   - Integration of retrieved information with large language models
   - Evaluation metrics and handling of hallucinations



branches run : 5
wall clock   : 5.6s

=== FINDINGS ===
## Retrieval mechanisms and vector database indexing
Retrieval mechanisms query a vector database to identify the most semantically relevant documents by comparing the embedding of the user's prompt against stored embeddings. These retrieved passages are then injected into the prompt to provide the large language model with specific, up-to-date context for generating an accurate response.

## Query transformation and semantic search strategies
Query transformation involves rewriting or expanding user inputs to better match the semantic meaning of the underlying documents, while semantic search uses vector embeddings to retrieve contextually relevant information based on meaning rather than exact keyword matches. These strategies enhance the quality of the retrieved context, enabling the generative model to produce more accurate and grounded responses.

## Context assembly and prompt construction
Retrieval-augmented generation firs

### Why `research_one` uses its own schema

Look again at `SubtopicState`. The worker cannot see `subtopics`, `findings` or
`report` - only what the `Send` handed it. That is deliberate:

- It makes each branch **independent** and trivially parallel-safe.
- It documents the worker's contract.
- It is the isolated-state pattern from notebook 03, applied per branch.

> **Pitfall:** a very common mistake is typing the worker as the *parent* state
> and then reading a parent key inside it. It will `KeyError`, because the `Send`
> payload - not the parent state - is what the worker receives.

### 8. Streaming a fan-out

With `stream_mode="updates"` you see each branch land as it completes.

In [10]:
for chunk in research_app.stream(
        {"topic": "vector databases", "subtopics": [], "findings": [], "report": ""},
        stream_mode="updates"):
    for node, upd in chunk.items():
        if node == "research_one":
            print("branch done ->", upd["findings"][0].splitlines()[0])
        elif node == "plan":
            print("planned", len(upd["subtopics"]), "branches")
        else:
            print("reduce done")

planner produced 5 subtopics:
   - Vector database architecture and indexing algorithms
   - Embedding generation and similarity search techniques
   - Scalability, performance optimization, and distributed systems
   - Integration with large language models and RAG pipelines
   - Data management, security, and operational best practices
planned 5 branches


branch done -> ## Embedding generation and similarity search techniques
branch done -> ## Scalability, performance optimization, and distributed systems
branch done -> ## Integration with large language models and RAG pipelines
branch done -> ## Vector database architecture and indexing algorithms
branch done -> ## Data management, security, and operational best practices


reduce done


### Recap

- `Send(node, payload)` creates a branch **at runtime**; return a list of them from
  a conditional edge.
- The payload is the worker's **entire state**, so give the worker its own schema.
- The collector key **must** have a reducer (`Annotated[list, operator.add]`).
- Guard against the empty fan-out; validate the plan before mapping.
- `Send` + subgraphs is how you build real map-reduce agents.

### Next

**[05_composed_pipeline](05_composed_pipeline.ipynb)** - subgraphs, isolated state,
and `Send` in one system.